# 古典模型：Colab GPU 試訓

在 Colab 選「執行階段 → 變更執行階段類型 → GPU」，依序執行。免費 GPU 的型號、記憶體及時限不保證足夠；本筆記本尚未在 GPU 實測。下載底座及安裝套件需網路與足夠磁碟。現有六筆訓練資料僅供流程試驗，不能證明模型可靠或移除 1899 年後知識。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "請先啟用 GPU 執行階段"
print("GPU:", torch.cuda.get_device_name(0), "VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
!git clone --branch work --single-branch https://github.com/andrewlai1209-byte/BooleLOT.git /content/BooleLOT
%cd /content/BooleLOT
!git rev-parse HEAD

## 安裝訓練依賴

依專案原有 Unsloth 範例安裝；版本或 GPU 相容性尚須實測。如安裝後要求重新啟動，請重啟並重新執行筆記本。

In [ ]:
!pip install -q --upgrade uv
!uv pip install --system -q "torch>=2.8.0" "triton>=3.4.0" torchvision bitsandbytes "transformers==4.56.2" "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" "unsloth[base] @ git+https://github.com/unslothai/unsloth" "git+https://github.com/triton-lang/triton.git@05b2c186c1b6c9a08375389d5efe9cb4c401c075#subdirectory=python/triton_kernels"
!uv pip install --system --no-deps "transformers==4.56.2" tokenizers "trl==0.22.2"

In [ ]:
!python -m training.prepare_corpus
!python -m training.train_classical --dry-run

## 試訓並保存

以下以少量資料試行 10 步，需下載約數十 GB 的底座；若顯存不足或執行階段中斷便無法完成。完成後請下載壓縮檔；它只是 LoRA 附加權重，仍需相容的底座。

In [ ]:
import subprocess
subprocess.run(["python", "-m", "training.train_classical", "--allow-small-dataset", "--max-steps", "10"], check=True)

In [ ]:
from pathlib import Path
import shutil
from google.colab import files
adapter = Path("data/classical/model-adapter")
assert (adapter / "adapter_config.json").exists(), "訓練尚未產生附加權重"
archive = shutil.make_archive("/content/classical-adapter", "zip", adapter)
files.download(archive)